In [2]:
import google.generativeai as genai
import json
import random
import time
from tqdm import tqdm

In [4]:
# ==========================================
# SYNTHETIC DATA GENERATOR 
# ==========================================


def get_read_heavy_profile():
    """Simulates a workload that needs high cache (shared_buffers)"""
    return {
        "input": (
            f"workload features: size: {random.randint(10, 50)}GB; read ratio: 0.9{random.randint(0, 9)}; "
            f"write ratio: 0.0{random.randint(1, 9)}; active_connections: {random.randint(50, 200)};\n"
            f"query plans: Seq Scan(cost={random.randint(500, 5000)}); Index Scan(cost={random.randint(10, 100)});\n"
            f"metrics: buffer_hit_ratio=0.99; disk_read_count={random.randint(100, 1000)}; lock_wait=0.0"
        ),
        "output": json.dumps({
            "shared_buffers": f"{random.randint(4, 8)}GB",
            "effective_cache_size": f"{random.randint(12, 24)}GB",
            "random_page_cost": "1.1", 
            "work_mem": "16MB"
        })
    }

def get_write_heavy_profile():
    """Simulates a transactional workload that needs WAL optimization"""
    return {
        "input": (
            f"workload features: size: {random.randint(50, 200)}GB; read ratio: 0.1{random.randint(0, 5)}; "
            f"write ratio: 0.8{random.randint(0, 9)}; active_connections: {random.randint(200, 500)};\n"
            f"query plans: Insert(cost={random.randint(10, 50)}); Update(cost={random.randint(20, 100)});\n"
            f"metrics: buffer_hit_ratio=0.45; wal_generation_rate={random.randint(10, 50)}MB/s; checkpoint_sync_time=200ms"
        ),
        "output": json.dumps({
            "shared_buffers": "2GB", 
            "max_wal_size": f"{random.randint(4, 16)}GB",
            "min_wal_size": "1GB",
            "checkpoint_timeout": "30min",
            "bgwriter_delay": "200ms"
        })
    }

def get_complex_analytical_profile():
    """Simulates complex joins that need high work_mem"""
    return {
        "input": (
            f"workload features: size: {random.randint(20, 100)}GB; read ratio: 0.95; "
            f"group_by_ratio: 0.8; order_by_ratio: 0.7; joins: {random.randint(3, 8)};\n"
            f"query plans: Hash Join(cost={random.randint(5000, 20000)}); Sort(cost={random.randint(2000, 10000)});\n"
            f"metrics: temp_file_written={random.randint(100, 500)}MB; buffer_hit_ratio=0.85"
        ),
        "output": json.dumps({
            "shared_buffers": "4GB",
            "work_mem": f"{random.randint(64, 256)}MB", 
            "random_page_cost": "1.1",
            "max_parallel_workers_per_gather": "4"
        })
    }

synthetic_data = []
for _ in range(20):
    synthetic_data.append(get_read_heavy_profile())
for _ in range(20):
    synthetic_data.append(get_write_heavy_profile())
for _ in range(10):
    synthetic_data.append(get_complex_analytical_profile())

random.shuffle(synthetic_data)

output_file = "e2etune_raw_data_synthetic.json"
with open(output_file, "w") as f:
    json.dump(synthetic_data, f, indent=4)

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

# Discover available models and pick one that supports text generation
try:
    models = list(genai.list_models())
    # Normalize names by stripping optional 'models/' prefix
    def norm(n):
        return n.split('/')[-1]
    available_names = {norm(m.name): m.name for m in models}

    preferred = [
        "gemini-1.5-flash-latest",
        "gemini-1.5-flash-001",
        "gemini-1.5-flash"
    ]
    # Choose preferred if available
    chosen_full_name = next((available_names.get(n) for n in preferred if n in available_names), None)
    if chosen_full_name is None:
        # Fallbacks
        fallbacks = [
            "gemini-1.5-pro-latest",
            "gemini-1.5-pro-001",
            "gemini-1.0-pro"
        ]
        chosen_full_name = next((available_names.get(n) for n in fallbacks if n in available_names), None)
    if chosen_full_name is None:
        # Last resort: pick any model containing 'gemini' and 'flash' or 'pro'
        names = list(available_names.values())
        candidates = [n for n in names if any(k in n for k in ["gemini", "flash", "pro"])]
        chosen_full_name = candidates[0] if candidates else None
    if chosen_full_name is None:
        raise RuntimeError("No supported Gemini text model found. Please check API access and model availability.")
except Exception as e:
    raise RuntimeError(f"Failed to list/select models: {e}")

model = genai.GenerativeModel(chosen_full_name)
print(f"Using model: {chosen_full_name}")

Using model: models/gemini-2.5-flash


In [7]:
input_file = "e2etune_raw_data_synthetic.json"
output_file = "e2etune_cot_augmented.jsonl"

with open(input_file, 'r') as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} samples for augmentation.")

Loaded 50 samples for augmentation.


In [11]:
augmented_dataset = []

# Simple backoff retry for transient errors
def generate_with_retry(prompt, max_retries=3, base_sleep=1.5):
    for attempt in range(1, max_retries + 1):
        try:
            response = model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            if attempt == max_retries:
                raise e
            time.sleep(base_sleep * attempt)
    return None

for entry in tqdm(raw_data):
    workload = entry['input']
    config = entry['output']
    
    # The Prompt: Ask Gemini to explain the link between Input -> Output
    prompt = f"""
    You are a PostgreSQL Database Tuning Expert.
    
    Analyze this Workload and the Config we chose for it.
    Explain the CAUSAL LOGIC (Chain of Thought) for why this config fits this workload.
    
    WORKLOAD: {workload}
    CONFIG: {config}
    
    OUTPUT: Provide ONLY the concise reasoning trace (2-3 sentences). Start with \"Reasoning:\".
    """
    
    try:
        reasoning = generate_with_retry(prompt)
        
        # 4. FORMAT FOR FINE-TUNING
        # We combine Reasoning + Config into the final target
        new_entry = {
            "instruction": f"Tune the database for this workload:\n{workload}",
            "output": f"{reasoning}\n\nFinal Config:\n```json\n{config}\n```"
        }
        augmented_dataset.append(new_entry)
        
        # Respect Free Tier limits
        time.sleep(1.5) 
        
    except Exception as e:
        print(f"Skipping row due to error: {e}")

  8%|▊         | 4/50 [00:49<09:24, 12.27s/it]



KeyboardInterrupt: 